© 2026 Mechanome Bio. Proprietary and confidential.

# Running BIDCell
repo: https://github.com/SydneyBioX/BIDCell

- pip package `bidcell`, Python >=3.9,<3.13. GPU needed (README: >=12 GB VRAM, 32 GB RAM).
- Config yaml is written below into `../config_files/`.
- Needs a single-cell reference, pre-made, in `../config_files/BIDCell_references/` (format as in the repo's `data/sc_references/`):
  - `reference.csv`: rows = cell types, columns = panel genes (mean expression), plus trailing columns `ct_idx`, `cell_type`, `atlas`
  - `positive_markers.csv` / `negative_markers.csv`: rows = cell types, columns = genes, values 0/1
- BIDCell segments nuclei itself (cellpose) from the DAPI image.

In [ ]:
import os
import sys

import numpy
import pandas
import tifffile
import yaml
from skimage.transform import resize

sys.path.append("..")
from utils import seg_utils

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
METHOD = "bidcell"
BOUNDING_BOX_MICRONS = (4673, 1476, 5036, 1805)  # microns; None = whole slide

REFERENCE_DIRECTORY = seg_utils.CONFIG_DIRECTORY / "BIDCell_references"
ELONGATED_CELL_TYPES = []  # names must match the 'cell_type' column of reference.csv, e.g. fibroblasts
WORK_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "BIDCell" / DATASET_NAME  # BIDCell's data_dir
WORK_DIRECTORY.mkdir(parents=True, exist_ok=True)

sdata = seg_utils.load_sdata(DATASET_NAME)

In [ ]:
seg_utils.preview_bounding_box(sdata, BOUNDING_BOX_MICRONS)

In [ ]:
sdata_crop = seg_utils.crop_sdata(sdata, BOUNDING_BOX_MICRONS)

## Inputs
DAPI tif (y, x) and a transcripts csv in microns relative to the crop's top-left, so the two line up at (0, 0).

In [ ]:
dapi = seg_utils.get_image(sdata_crop)
tifffile.imwrite(WORK_DIRECTORY / "dapi.tif", dapi)

transcripts = seg_utils.get_transcripts(sdata_crop)
transcripts_for_bidcell = pandas.DataFrame({
    "x_location": transcripts["x_pixels"] * seg_utils.PIXEL_SIZE_MICRONS,
    "y_location": transcripts["y_pixels"] * seg_utils.PIXEL_SIZE_MICRONS,
    "feature_name": transcripts[seg_utils.GENE_COLUMN].astype(str),
    "qv": transcripts["qv"],
})
transcripts_for_bidcell.to_csv(WORK_DIRECTORY / "transcripts.csv.gz", index=False)

## Config
Based on the repo's `example_params/xenium.yaml`.

In [ ]:
config = {
    "cpus": os.cpu_count(),
    "files": {
        "data_dir": str(WORK_DIRECTORY),
        "fp_dapi": str(WORK_DIRECTORY / "dapi.tif"),
        "fp_transcripts": str(WORK_DIRECTORY / "transcripts.csv.gz"),
        "fp_ref": str(REFERENCE_DIRECTORY / "reference.csv"),
        "fp_pos_markers": str(REFERENCE_DIRECTORY / "positive_markers.csv"),
        "fp_neg_markers": str(REFERENCE_DIRECTORY / "negative_markers.csv"),
    },
    "nuclei_fovs": {"stitch_nuclei_fovs": False},
    "nuclei": {"diameter": None},  # None = cellpose estimates it
    "transcripts": {
        "shift_to_origin": False,
        "x_col": "x_location",
        "y_col": "y_location",
        "gene_col": "feature_name",
        "transcripts_to_filter": ["NegControlProbe_", "antisense_", "NegControlCodeword_", "BLANK_", "Blank-", "NegPrb",
                                  "DeprecatedCodeword_", "UnassignedCodeword_"],
    },
    "affine": {
        "target_pix_um": 1.0,
        "base_pix_x": seg_utils.PIXEL_SIZE_MICRONS,
        "base_pix_y": seg_utils.PIXEL_SIZE_MICRONS,
        "base_ts_x": 1.0,
        "base_ts_y": 1.0,
        "global_shift_x": 0,
        "global_shift_y": 0,
    },
    "model_params": {"name": "custom", "patch_size": 48, "elongated": ELONGATED_CELL_TYPES},
    "training_params": {"total_epochs": 1, "total_steps": 4000, "ne_weight": 1.0, "os_weight": 1.0,
                        "cc_weight": 1.0, "ov_weight": 1.0, "pos_weight": 1.0, "neg_weight": 1.0},
    "testing_params": {"test_epoch": 1, "test_step": 4000},
    "experiment_dirs": {"dir_id": "last"},
}

config_path = seg_utils.CONFIG_DIRECTORY / f"BIDCell_{DATASET_NAME}_config.yaml"
with open(config_path, "w") as file:
    file.write(f"# {seg_utils.NOTICE}\n")
    yaml.safe_dump(config, file, sort_keys=False)

## Run

In [ ]:
from bidcell import BIDCellModel

model = BIDCellModel(str(config_path))
model.preprocess()
model.train()
model.predict()

## Save
Final mask: `model_outputs/<timestamp>/test_output/epoch_*_step_*_connected.tif`, at `target_pix_um` (1 µm) resolution,
so it's resized (nearest neighbour) back to the DAPI grid first.

In [ ]:
mask_path = sorted(WORK_DIRECTORY.glob("model_outputs/*/test_output/*_connected.tif"))[-1]
mask = tifffile.imread(mask_path)
mask = resize(mask, dapi.shape, order=0, preserve_range=True, anti_aliasing=False).astype(numpy.int32)

polygons = seg_utils.mask_to_polygons(mask, origin_pixels=seg_utils.image_origin_pixels(sdata_crop))
seg_utils.save_segmentation(sdata, polygons, METHOD, DATASET_NAME, BOUNDING_BOX_MICRONS)